# Fine-tune best.pt on original + 3k football dataset

## 1. Check GPU & install

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics roboflow

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Setup: mount Drive, config

In [2]:
import os
import shutil
from pathlib import Path
import yaml
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/football-finetune')
RUNS_DIR = DRIVE_DIR / 'runs'          # checkpoints go to Drive (survive disconnects)

WORK = Path('/content/work')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
shutil.copy(DRIVE_DIR / 'best.pt', WORK / 'best.pt')
BASE_MODEL = str(WORK / 'best.pt')

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')  # set in Colab Secrets panel

TARGET = ['ball', 'goalkeeper', 'player', 'referee']   # best.pt class order
# Alternate class names other datasets may use -> TARGET names.
ALIASES = {'football': 'ball', 'soccer-ball': 'ball', 'soccerball': 'ball', 'sports ball': 'ball',
           'players': 'player', 'goalkeepers': 'goalkeeper', 'referees': 'referee'}

DATASETS = {
    # (workspace, project, version) - check the version number on each Universe page
    'original': ('roboflow-jvuqo', 'football-players-detection-3zvbc', 1),
    'label3k':  ('label-gdoal', 'soccer-oilql', 1),
}
MERGED = WORK / 'merged_dataset'
RUN_NAME = 'finetune_label3k'

Mounted at /content/drive


## 3. Download datasets

In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
paths = {}
for key, (ws, proj, ver) in DATASETS.items():
    project = rf.workspace(ws).project(proj)
    version = project.version(ver)
    dataset = version.download('yolov8', location=str(WORK / 'raw' / key))
    paths[key] = Path(dataset.location)
paths

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/work/raw/original in yolov8:: 100%|██████████| 1338/1338 [00:00<00:00, 2259.53it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/work/raw/label3k in yolov8:: 100%|██████████| 6012/6012 [00:00<00:00, 6870.94it/s]


{'original': PosixPath('/content/work/raw/original'),
 'label3k': PosixPath('/content/work/raw/label3k')}

## 4. Merge datasets (remap classes by name)

Class IDs are remapped **by name** so every dataset matches best.pt's order.
Check the printed mapping: all 4 classes should appear; anything "dropping" that is really
a ball/player/etc. should be added to `ALIASES` in the setup cell.

The **original test split is kept as the benchmark**; the 3k set's test images go into training.

In [4]:
def class_map(ds_dir):
    names = yaml.safe_load((ds_dir / 'data.yaml').read_text())['names']
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    mapping = {}
    for i, n in enumerate(names):
        key = ALIASES.get(n.lower().strip(), n.lower().strip())
        if key in TARGET:
            mapping[i] = TARGET.index(key)
        else:
            print(f'  [{ds_dir.name}] dropping unknown class {n!r}')
    print(f'  [{ds_dir.name}] {names} -> {mapping}')
    return mapping


def copy_split(src, dst_split, mapping, prefix):
    img_dir, lbl_dir = src / 'images', src / 'labels'
    if not img_dir.exists():
        return 0
    out_img = MERGED / dst_split / 'images'
    out_lbl = MERGED / dst_split / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    n = 0
    for img in img_dir.iterdir():
        lbl = lbl_dir / f'{img.stem}.txt'
        lines = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if parts and int(parts[0]) in mapping:
                    lines.append(' '.join([str(mapping[int(parts[0])])] + parts[1:]))
        shutil.copy(img, out_img / f'{prefix}_{img.name}')
        (out_lbl / f'{prefix}_{img.stem}.txt').write_text('\n'.join(lines))
        n += 1
    return n


if MERGED.exists():
    shutil.rmtree(MERGED)
for key, d in paths.items():
    m = class_map(d)
    if key == 'original':
        splits = {'train': 'train', 'valid': 'valid', 'test': 'test'}
    else:
        splits = {'train': 'train', 'valid': 'valid', 'test': 'train'}
    for s, dst in splits.items():
        print(f'  {key}/{s} -> {dst}: {copy_split(d / s, dst, m, key)} images')

DATA_YAML = MERGED / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(MERGED),
    'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
    'names': dict(enumerate(TARGET)),
}))
for s in ['train', 'valid', 'test']:
    print(s, len(list((MERGED / s / 'images').iterdir())), 'images')

  [original] ['ball', 'goalkeeper', 'player', 'referee'] -> {0: 0, 1: 1, 2: 2, 3: 3}
  original/train -> train: 612 images
  original/valid -> valid: 38 images
  original/test -> test: 13 images
  [label3k] ['ball', 'goalkeeper', 'player', 'referee'] -> {0: 0, 1: 1, 2: 2, 3: 3}
  label3k/train -> train: 1500 images
  label3k/valid -> valid: 750 images
  label3k/test -> train: 750 images
train 2862 images
valid 788 images
test 13 images


## 5. Fine-tune

Low learning rate: we adapt the already-trained model, not retrain it.
If you get CUDA out-of-memory, lower `batch` to 8.

In [6]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=20,
    imgsz=640,          # try 1280 (with batch=4-8) if the ball class stays weak
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=1,
    patience=15,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
NEW_WEIGHTS = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'

Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/work/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/work/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=finetune_label3k, nbs=64, nms=None, ops

## 6. Compare old vs new on the original test split

The test split contains only the original dataset's test images, so this is a fair comparison
on footage like yours. Keep the new model only if it scores higher.

In [7]:
import pandas as pd

rows = []
for name, w in [('old', BASE_MODEL), ('new', str(NEW_WEIGHTS))]:
    r = YOLO(w).val(data=str(DATA_YAML), split='test', imgsz=640, verbose=False)
    row = {'model': name, 'mAP50': r.box.map50, 'mAP50-95': r.box.map}
    row.update({f'{c} mAP50-95': v for c, v in zip(TARGET, r.box.maps)})
    rows.append(row)
pd.DataFrame(rows).set_index('model').round(3)

Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv5l6u summary (fused): 167 layers, 85,975,392 parameters, 0 gradients, 137.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 105.5±77.6 MB/s, size: 225.1 KB)
val: Scanning /content/work/merged_dataset/test/labels... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 539.2it/s 0.0s
val: New cache created: /content/work/merged_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.7it/s 0.6s
                   all         13        310      0.923      0.782      0.813       0.57
Speed: 0.3ms preprocess, 29.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/work/runs/detect/val
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv5l6u summary (fused): 167 layers, 85,975,392 parameters, 0 gradients, 137.0 GFLOPs
val: Fast image access ✅

,mAP50,mAP50-95,ball mAP50-95,goalkeeper mAP50-95,player mAP50-95,referee mAP50-95
model,,,,,,
old,0.813,0.570,0.161,0.772,0.759,0.589
new,0.810,0.581,0.153,0.773,0.756,0.641


In [8]:
print("Old:", YOLO(BASE_MODEL).names)
print("New:", YOLO(str(NEW_WEIGHTS)).names)
print("Target:", dict(enumerate(TARGET)))

Old: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
New: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
Target: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}


In [10]:
old_model = YOLO(BASE_MODEL)
new_model = YOLO(str(NEW_WEIGHTS))

In [11]:
# old_model.predict("/content/full.mp4", save = True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/750) /content/full.mp4: 384x640 1 ball, 1 goalkeeper, 18 players, 4 referees, 46.9ms
video 1/1 (frame 2/750) /content/full.mp4: 384x640 1 ball, 1 goalkeeper, 18 players, 4 referees, 40.8ms
video 1/1 (frame 3/750) /content/full.mp4: 384x640 1 goalkeeper, 18 players, 4 referees, 45.5ms
video 1/1 (frame 4/750) /content/full.mp4: 384x640 1 ball, 1 goalkeeper, 19 players, 4 referees, 40.0ms
video 1/1 (frame 5/750) /content/full.mp4: 384x640 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 colors: None
 depth: None
 keypoints: None
 masks: None
 names: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
 obb: None
 orig_img: array([[[100, 146, 105],
         [ 92, 138,  97],
         [ 97, 150, 101],
         ...,
         [100,  92,  82],
         [103,  95,  85],
         [105,  97,  87]],
 
        [[ 99, 145, 104],
         [ 99, 145, 104],
         [109, 162, 113],
         ...,
         [105,  97,  87],
         [107,  99,  89],
         [108, 100,  90]],
 
        [[ 96, 149, 100],
         [105, 158, 109],
         [112, 170, 113],
         ...,
         [106,  98,  88],
         [108, 100,  90],
         [110, 102,  92]],
 
        ...,
 
        [[ 74, 103,  78],
         [ 74, 103,  78],
         [ 74, 103,  78],
         ...,
         [ 30,  47,  43],
         [ 31,  48,  44],
         [ 31,  48,  44]],
 
        [[ 74, 103,  78],
         [ 74, 

## 7. Download the new weights
Put it in your project as `models/best_finetuned.pt`.

In [9]:
# from google.colab import files

# files.download(str(NEW_WEIGHTS))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>